# LISTA 09 - SISTEMAS LIENARES: MÉTODOS ITERATIVOS

Aluna: Maria Eduarda Saldanha Alves

## Exercício 1

Considere o código de geração de sistemas lineares mostrado no Apêndice A e faça o seguinte:

- Gere 10 sistemas lineares distintos usando sua matrícula como semente acrescida de um múltiplo de $100$.
- Por exemplo, para uma matrícula igual a $5000$, podemos usar como semente os números $5000$, $5100$, $5200$, ...
- Salve cada sistema gerado em um arquivo CSV, pode ser usado valor da semente como nome do arquivo.
- A Figura 1 mostra um exemplo de utilização do gerador, usando a semente $10$ e salvando o arquivo como `10.csv`.
- Para cada sistema gerado, faça o teste de convergência e execute os métodos Jacobi e Gauss-Seidel para resolvê-los.
- Utilize zero como valor inicial para todas as variáveis, tolerância de $10^{-5}$ e o máximo de $50$ iterações.
- O Apêndice B mostra como deve ser a resposta de execução dos métodos.

## GERAÇÃO DOS SISTEMAS

In [37]:
import os

# apaga os CSVs antigos
for f in os.listdir():
    if f.endswith('.csv'):
        os.remove(f)

matricula = 118092
sementes = [matricula + 100 * i for i in range(10)]
print("Sementes utilizadas:", sementes)

for s in sementes:
    !python gerador.py -p4 -s{s} -a {s}.csv

print("\nArquivos gerados:")
print([f for f in os.listdir() if f.endswith('.csv')])

Sementes utilizadas: [118092, 118192, 118292, 118392, 118492, 118592, 118692, 118792, 118892, 118992]

Arquivos gerados:
['118092.csv', '118192.csv', '118292.csv', '118392.csv', '118492.csv', '118592.csv', '118692.csv', '118792.csv', '118892.csv', '118992.csv']


## DEFININDO AS FUNÇÕES E UTILIZANDO OS SISTEMAS

In [38]:
import numpy as np
import os

# --- funções dos métodos  ---
def jacobi(A, b, x0, tol, n):
    d = len(A)
    x = np.copy(x0)
    xant = np.copy(x0)
    for k in range(n):
        for i in range(d):
            x[i] = b[i]
            for j in range(d):
                if j != i:
                    x[i] = x[i] - A[i, j] * xant[j]
            x[i] = x[i] / A[i, i]
        erro = np.max(np.abs(x - xant))
        if erro < tol:
            break
        xant = np.copy(x)
    return x, k + 1, erro 

def gauss_seidel(A, b, x0, tol, n):
    d = len(A)
    x, x_ = np.copy(x0), np.copy(x0)
    for k in range(n):
        for i in range(d):
            x[i] = b[i]
            for j in range(i):
                x[i] = x[i] - A[i][j] * x[j]
            for j in range(i + 1, d):
                x[i] = x[i] - A[i][j] * x_[j]
            x[i] = x[i] / A[i, i]
        erro = np.max(np.abs(x - x_))
        if erro < tol:
            break
        x_ = np.copy(x)
    return x, k + 1, erro

# --- loop principal do exercício ---
matricula = 118092
sementes = [matricula + 100 * i for i in range(10)]
tol = 1e-5
max_iter = 50

for s in sementes:
    arquivo = f"{s}.csv"
    if not os.path.exists(arquivo):
        print(f"Arquivo {arquivo} não encontrado. Pulando...")
        continue

    # carrega o sistema
    mat = np.loadtxt(arquivo, delimiter=',')
    A = mat[:, :-1]
    b = mat[:, -1]
    n = len(A)
    x0 = np.zeros(n)

    # --- teste de Convergência ---
    D = np.diag(np.diag(A))
    L = np.tril(A) - D
    U = np.triu(A) - D

    TJ = -np.linalg.inv(D).dot(L + U)
    raio_j = max(abs(np.linalg.eig(TJ)[0]))
    converge_j = raio_j < 1

    TG = -np.linalg.inv(L + D).dot(U)
    raio_g = max(abs(np.linalg.eig(TG)[0]))
    converge_g = raio_g < 1

    # --- executando os métodos ---
    print(f"\n{'='*60}")
    print(f"SISTEMA: {arquivo} (Semente {s})")
    print(f"{'='*60}")
    print(f"Raio Espectral Jacobi:      {raio_j:.6f} -> Converge? {converge_j}")
    print(f"Raio Espectral Gauss-Seidel: {raio_g:.6f} -> Converge? {converge_g}")
    
    if converge_j:
        sol_j, iter_j, erro_j = jacobi(A, b, x0, tol, max_iter)
        print(f"\nJacobi: parou na iteração {iter_j} com erro {erro_j:.2e}")
        print(f"Solução: {np.round(sol_j, 4)}")
    else:
        print("\nJacobi: Não converge.")

    if converge_g:
        sol_g, iter_g, erro_g = gauss_seidel(A, b, x0, tol, max_iter)
        print(f"\nGauss-Seidel: parou na iteração {iter_g} com erro {erro_g:.2e}")
        print(f"Solução: {np.round(sol_g, 4)}")
    else:
        print("\nGauss-Seidel: Não converge.")


SISTEMA: 118092.csv (Semente 118092)
Raio Espectral Jacobi:      0.477892 -> Converge? True
Raio Espectral Gauss-Seidel: 0.316571 -> Converge? True

Jacobi: parou na iteração 19 com erro 6.96e-06
Solução: [-7.0714  6.6673  0.6061 -5.8592  7.     -1.0102]

Gauss-Seidel: parou na iteração 12 com erro 7.65e-06
Solução: [-7.0714  6.6673  0.6061 -5.8592  7.     -1.0102]

SISTEMA: 118192.csv (Semente 118192)
Raio Espectral Jacobi:      0.600473 -> Converge? True
Raio Espectral Gauss-Seidel: 0.534523 -> Converge? True

Jacobi: parou na iteração 28 com erro 7.70e-06
Solução: [-5.051   2.6265 -8.2837 -3.      8.2836]

Gauss-Seidel: parou na iteração 24 com erro 5.57e-06
Solução: [-5.051   2.6265 -8.2837 -3.      8.2836]

SISTEMA: 118292.csv (Semente 118292)
Raio Espectral Jacobi:      0.493092 -> Converge? True
Raio Espectral Gauss-Seidel: 0.258422 -> Converge? True

Jacobi: parou na iteração 14 com erro 6.64e-06
Solução: [ -9.      -7.0714  -5.4551   6.6673   3.4347 -10.      -4.6469]

Gauss-

## CONCLUSÃO

### Análise dos Resultados

Foram gerados **10 sistemas lineares** distintos usando as sementes `118092, 118192, ..., 118992`.

Para **todos os 10 sistemas**, os raios espectrais de ambas as matrizes de iteração foram **menores que 1**, o que garante a convergência dos métodos de Jacobi e Gauss-Seidel.

### Comparação entre os métodos

| Sistema   | Iter. Jacobi | Iter. Gauss-Seidel | Erro Jacobi | Erro Gauss-Seidel |
|-----------|:------------:|:------------------:|:-----------:|:-----------------:|
| 118092    | 19           | 12                 | 6,96e-06    | 7,65e-06          |
| 118192    | 28           | 24                 | 7,70e-06    | 5,57e-06          |
| 118292    | 14           | 11                 | 6,64e-06    | 6,95e-06          |
| 118392    | 16           | 10                 | 4,12e-06    | 6,75e-06          |
| 118492    | 13           | 9                  | 5,75e-06    | 2,09e-06          |
| 118592    | 16           | 11                 | 4,89e-06    | 8,85e-06          |
| 118692    | 15           | 10                 | 7,37e-06    | 9,69e-06          |
| 118792    | 19           | 8                  | 6,70e-06    | 8,79e-06          |
| 118892    | 17           | 10                 | 6,40e-06    | 7,33e-06          |
| 118992    | 15           | 9                  | 8,15e-06    | 5,73e-06          |

### Conclusões

1. **Convergência garantida:** Em todos os casos, os raios espectrais foram menores que 1, confirmando a convergência teórica dos dois métodos.
2. **Gauss-Seidel é mais eficiente:** Em **todos** os 10 sistemas, o Gauss-Seidel precisou de **menos iterações** que o Jacobi para atingir a tolerância de `10⁻⁵`, o que está de acordo com a teoria (o raio espectral do Gauss-Seidel foi sempre menor que o do Jacobi).
3. **Mesma solução:** Ambos os métodos convergiram para a mesma solução em cada sistema, diferindo apenas a partir da 5ª ou 6ª casa decimal.